In [15]:
import pandas as pd
import re

In [16]:
df1 = pd.read_excel("F:/Payments Reconciliation/Payments Reconciliation/MPESA_B2C_TRANSACTION.xlsx")
df2 = pd.read_excel("F:/Payments Reconciliation/Payments Reconciliation/HOLD TRANSACTIONS WITH DATETIME.xlsx")

In [17]:
print(df1.columns)
print(df2.columns)

Index(['SOURCE_TYPE', 'SOURCE_ID', 'SOURCE_NO', 'SR_NO', 'SUBMIT_DATE',
       'SUBMIT_TIME', 'PAYEE_MOBILE_NO', 'CORP_CODE', 'AMOUNT',
       'CONVERSATION_ID', 'ORIGINAL_CONVERSATION_ID', 'SUBMIT_STATUS',
       'TRANS_NO', 'TRANSFER_DATE', 'TRANSFER_TIME', 'TRANSFER_STATUS',
       'TRANSFER_REF_NO', 'HOLD_SR_NO'],
      dtype='object')
Index(['S/NO', 'CRO', 'customer_name', 'phone_number', 'EMAIL', 'SOURCE_ID',
       'SOURCE_NO', 'AMOUNT', 'mpesa_amount', 'payment_timestamp'],
      dtype='object')


In [18]:
def clean_SourceNo(df, col="SOURCE_NO"):
    
   """ Standardize SourceNo column:
    - Convert to string
    - Remove .0 from Excel numbers
    - Strip leading/trailing spaces
    - Remove non-breaking spaces """

   df[col] = (
        df[col]
        .astype(str)                     # force string
        .str.replace(".0", "", regex=False)  # remove .0 from Excel
        .str.strip()                     # strip spaces
        .str.replace("\u00A0", "", regex=True) # remove non-breaking space
    )
   
   return df
df1 = clean_SourceNo(df1, "SOURCE_NO")
df2 = clean_SourceNo(df2, "SOURCE_NO")

In [19]:
"""def normalize_account(x):
    x = str(x)

    if "/" in x:
        return x.split("/")[-1].lstrip("0")

    match = re.search(r"[A-Za-z]+0*(\d+)", x)
    if match:
        return match.group(1)

    return x

df1["MemberKey"] = df1["MemberNo"].apply(normalize_account)
df2["MemberKey"] = df2["MemberNo"].apply(normalize_account)
"""

<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Joy.Ireri\AppData\Local\Temp\ipykernel_24220\2492740644.py:7: SyntaxWarning: invalid escape sequence '\d'
  match = re.search(r"[A-Za-z]+0*(\d+)", x)


'def normalize_account(x):\n    x = str(x)\n\n    if "/" in x:\n        return x.split("/")[-1].lstrip("0")\n\n    match = re.search(r"[A-Za-z]+0*(\\d+)", x)\n    if match:\n        return match.group(1)\n\n    return x\n\ndf1["MemberKey"] = df1["MemberNo"].apply(normalize_account)\ndf2["MemberKey"] = df2["MemberNo"].apply(normalize_account)\n'

In [20]:
# print(type(df1["SOURCE_NO"].iloc[0]))


In [21]:
df1_subset = df1[["SOURCE_NO", "AMOUNT","TRANSFER_REF_NO", "TRANSFER_STATUS"]]

"""df1_subset_first = (
    df1_subset
    .drop_duplicates(subset="SOURCE_NO", keep="first")
)
merged_df = df2.merge(
    df1_subset,
    on="SOURCE_NO",
    how="left"
)"""

'df1_subset_first = (\n    df1_subset\n    .drop_duplicates(subset="SOURCE_NO", keep="first")\n)\nmerged_df = df2.merge(\n    df1_subset,\n    on="SOURCE_NO",\n    how="left"\n)'

In [22]:
df1_subset["dup_key"] = df1_subset.groupby(
    ["SOURCE_NO", "AMOUNT"]
).cumcount()

df2["dup_key"] = df2.groupby(
    ["SOURCE_NO", "AMOUNT"]
).cumcount()

merged_df = df2.merge(
    df1_subset,
    on=["SOURCE_NO", "AMOUNT", "dup_key"],
    how="left",
    indicator=True
)

print(merged_df[["_merge"]].value_counts())

C:\Users\Joy.Ireri\AppData\Local\Temp\ipykernel_24220\2910168035.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_subset["dup_key"] = df1_subset.groupby(


_merge    
both          2450
left_only      242
right_only       0
Name: count, dtype: int64


In [23]:
with pd.ExcelWriter("F:/Payments Reconciliation/Payments Reconciliation/MPESA B2C WITH REF8.xlsx",engine = "xlsxwriter")as writer:
    merged_df.to_excel(writer, sheet_name = "summary",index=False)